In [ ]:
import json
import os
import random
import re
import time
from datetime import datetime
from typing import Dict, List, Union, Tuple, Set, Optional
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import gseapy as gp
import pandas as pd
import requests
from tqdm import tqdm
from typing import List, Dict, Any, Optional, Tuple
from rapidfuzz import process

In [ ]:
# =============================================================================
# BREAST CANCER — TP53 MUT vs WT
# TOP 50 UPREGULATED PROTEINS
# =============================================================================

import pandas as pd
import numpy as np
from scipy import stats

FILE = "BC_metadata_41467_2019_9018_MOESM3_ESM (1).xlsx"

meta = pd.read_excel(
    FILE,
    sheet_name="Tumor annotations",
    header=0
)

# Keep OSL tumor samples
meta = meta[
    meta["Tumor ID"].astype(str).str.startswith("OSL")
].copy()

meta = meta.set_index("Tumor ID")

meta.index = meta.index.astype(str).str.strip()

print("Metadata shape:", meta.shape)
print("TP53 unique values:", meta["TP53"].unique())

prot_raw = pd.read_excel(
    FILE,
    sheet_name="1 Prot FDR 9995 quant",
    header=None
)

# 45 sample columns
ratio_col_indices = list(range(34, 79))

sample_ids = (
    prot_raw.iloc[0, ratio_col_indices]
    .astype(str)
    .str.strip()
    .tolist()
)

gene_symbols = (
    prot_raw.iloc[3:, 0]
    .astype(str)
    .str.strip()
)

ratio_data = prot_raw.iloc[
    3:,
    ratio_col_indices
].copy()

prot = ratio_data.copy()

prot.columns = sample_ids
prot.index = gene_symbols.values

prot = prot.apply(
    pd.to_numeric,
    errors="coerce"
)

prot = prot[
    ~prot.index.isin(["nan", "", "NaN"])
]

print("Proteomics shape:", prot.shape)

print(prot.iloc[:3, :4])



# Replace zero with missing
prot = prot.replace(0, np.nan)

# Remove any remaining non-positive values
prot[prot <= 0] = np.nan

# Log2 transform
prot = np.log2(prot)

print("\nLog2 transform applied.")


wt_samples = meta[
    meta["TP53"] == "wt"
].index.tolist()

mut_samples = meta[
    meta["TP53"] == "mut"
].index.tolist()


wt_samples = [
    s for s in wt_samples
    if s in prot.columns
]

mut_samples = [
    s for s in mut_samples
    if s in prot.columns
]

print(f"\nWT samples:  {len(wt_samples)}")
print(f"MUT samples: {len(mut_samples)}")


wt = prot[wt_samples]
mut = prot[mut_samples]


results = []

for protein in prot.index:

    wt_vals = wt.loc[protein].dropna()
    mut_vals = mut.loc[protein].dropna()

    if len(wt_vals) < 2 or len(mut_vals) < 2:
        continue

    t_stat, p_val = stats.ttest_ind(
        mut_vals,
        wt_vals,
        equal_var=False
    )

    mean_wt = wt_vals.mean()
    mean_mut = mut_vals.mean()

    log2fc = mean_mut - mean_wt

    results.append({
        "protein": protein,
        "mean_WT": mean_wt,
        "mean_MUT": mean_mut,
        "log2FC": log2fc,
        "t_stat": t_stat,
        "p_value": p_val,
        "n_WT": len(wt_vals),
        "n_MUT": len(mut_vals)
    })


results_df = pd.DataFrame(results)

results_df = results_df.replace(
    [np.inf, -np.inf],
    np.nan
)

results_df = results_df.dropna(
    subset=["p_value", "log2FC"]
)

print(
    f"\nProteins tested: {len(results_df)}"
)


sig = results_df[
    results_df["p_value"] < 0.01
].copy()

print(
    f"Proteins with raw p < 0.01: {len(sig)}"
)



upregulated = sig[
    sig["log2FC"] > 0
].copy()

print(
    f"TP53 MUT upregulated proteins with raw p < 0.01: "
    f"{len(upregulated)}"
)


upregulated["neg_log10_p"] = (
    -np.log10(upregulated["p_value"])
)

upregulated["ranking_score"] = (
    upregulated["log2FC"]
    * upregulated["neg_log10_p"]
)

top50_up = (
    upregulated
    .sort_values(
        "ranking_score",
        ascending=False
    )
    .head(50)
    .copy()
)

# Add rank
top50_up.insert(
    0,
    "Rank",
    range(1, len(top50_up) + 1)
)


print("\n")
print("=" * 100)
print("TOP 50 TP53 MUT-UPREGULATED PROTEINS")
print("=" * 100)

print(
    top50_up[
        [
            "Rank",
            "protein",
            "mean_WT",
            "mean_MUT",
            "log2FC",
            "p_value",
            "neg_log10_p",
            "ranking_score",
            "n_WT",
            "n_MUT"
        ]
    ].to_string(index=False)
)


top50_up.to_csv(
    "BreastCancer_TP53_Top50_Upregulated.csv",
    index=False
)

print(
    "\nSaved table to: "
    "BreastCancer_TP53_Top50_Upregulated.csv"
)

top50_genes = top50_up[
    "protein"
].tolist()

with open(
    "BreastCancer_TP53_RefinedCommunity1.txt",
    "w"
) as f:

    f.write(
        f"Refined Community 1: {top50_genes}\n"
    )

print(
    "\nSaved gene list to: "
    "BreastCancer_TP53_RefinedCommunity1.txt"
)

print(
    "\nRefined Community 1:"
)

print(top50_genes)

In [ ]:
class OptimizedPathwayAnalyzer:
    def __init__(self):
        self.api_key = None
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.checkpoint_file = f"checkpoint_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        self.results_csv = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        self.pathway_gene_mapping = {}
        
    def set_api_key(self, api_key: str = None):
        """
        Set the Deepseek API key
        """
        if api_key:
            self.api_key = api_key
        elif os.environ.get("DEEPSEEK_API_KEY"):
            self.api_key = os.environ.get("DEEPSEEK_API_KEY")
        else:
            raise ValueError("No Deepseek API key provided. Set it via argument or DEEPSEEK_API_KEY environment variable.")
    
    def _create_system_prompt(self) -> str:
        return """ You are a bioinformatics expert. Focus on biological pathways associated with TP53 mutation status (upregulated) in Breast Cancer. Provide detailed pathway analysis with scientific literature support. Always cite specific papers when discussing pathway relationships."""
    
    def _create_all_in_one_prompt(self, genes: List[str], enrichment_pathways: List[str]) -> str:
        enrichment_section = ""
        if enrichment_pathways and len(enrichment_pathways) > 0:
            enrichment_section = "\nEnrichment Analysis Results:\n" + "\n".join(f"- {pathway}" for pathway in enrichment_pathways)
        
        genes_str = ", ".join(genes)
        
        enrichment_instruction = "Using the enrichment pathways above (if any), provide an analysis of the gene set"
        
        return f"""You are a bioinformatics expert conducting pathway analysis of gene sets. For the gene set: [{genes_str}]{enrichment_section}

Please perform analysis in clearly labeled sections:

===== SECTION 1: ANALYSIS WITH ENRICHMENT =====
{enrichment_instruction}:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. Provide explicit reasoning for choosing this pathway/process
4. List the specific genes from the gene set that contribute to the process
5. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
6. If there isn't sufficient evidence or fewer than two genes are associated with a common pathway, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITH ENRICHMENT: [Process Name] ([Confidence Score])

PATHWAY REASONING WITH ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

CONTRIBUTING GENES WITH ENRICHMENT:
[Comma-separated list of contributing genes]

ANALYSIS TEXT WITH ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 2: ANALYSIS WITHOUT ENRICHMENT =====
Based SOLELY on your knowledge of these genes, without considering the enrichment results:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. List the specific genes from the gene set that are involved in the process
4. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
5. IMPORTANT: If there isn't sufficient evidence or fewer than two genes share a common biological function, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITHOUT ENRICHMENT: [Process Name] ([Confidence Score])

CONTRIBUTING GENES WITHOUT ENRICHMENT:
[Comma-separated list of contributing genes]

PATHWAY REASONING WITHOUT ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

ANALYSIS TEXT WITHOUT ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 3: FINAL PROCESS SELECTION =====
Compare both analyses and provide:
1. Which process (with or without enrichment) has higher confidence and why
2. Final reasoning for the chosen process

Output format for this section:
FINAL PROCESS REASONING:
[Detailed explanation of why you chose the final process, comparing both analyses and explaining which one provides stronger evidence]

Analytical Guidelines:
- Be concise and avoid unnecessary words
- Be factual without editorializing
- Be specific, avoiding overly general statements
- Avoid listing individual protein facts
- Group proteins by similar functions
- Discuss their interplay, synergistic or antagonistic effects
- Focus on functional integration within the system
- Include at least 2-3 specific paper citations when discussing established pathway relationships

Confidence Score Instructions:
- Assign a score from 0.00 to 1.00
- 0.00 indicates lowest confidence
- 1.00 reflects highest confidence
- Base the score on the proportion of genes participating in the identified process"""
    
    def _make_deepseek_request(self, prompt: str) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        data = {
            "model": "deepseek-chat",
            "messages": [
                {"role": "system", "content": self._create_system_prompt()},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0,
            "max_tokens": 4000
        }
        
        response = requests.post(self.api_url, headers=headers, json=data)
        
        if response.status_code == 200:
            result = response.json()
            return result['choices'][0]['message']['content']
        else:
            raise Exception(f"Deepseek API request failed: {response.status_code} - {response.text}")
    
    def parse_communities(self, input_text: str) -> Dict[str, List[str]]:
        communities = {}
        for line in input_text.strip().split('\n'):
            line = line.strip()
            if 'Refined Community' in line:
                match = re.search(r'Refined Community (\d+): \[(.*?)\]', line)
                if match:
                    community_num = match.group(1)
                    genes_str = match.group(2)
                    genes = [g.strip("'") for g in genes_str.split(', ')]
                    communities[community_num] = genes
        return communities
    
    def perform_enrichment(self, gene_list: List[str]) -> List[str]:

        try:
            databases = ['GO_Biological_Process_2021', 'Reactome_2022', 'KEGG_2021_Human']
            
            all_results = []
            for database in databases:
                enr = gp.enrichr(gene_list=gene_list,
                                gene_sets=[database],
                                organism='Human',
                                outdir=None,
                                no_plot=True,
                                cutoff=0.01)
                
                results_df = enr.results
                if not results_df.empty:
                    results_df = results_df.sort_values('Adjusted P-value')
                    for _, row in results_df.head(5).iterrows():
                        all_results.append(row['Term'])
            
            return list(set(all_results))
        
        except Exception as e:
            print(f"Enrichment analysis failed: {str(e)}")
            return []
    
    def _clean_markdown_formatting(self, text: str) -> str:

        if not text or not isinstance(text, str):
            return text
            
        text = re.sub(r'\*([^*]+)\*', r'\1', text)
        
        text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
        
        text = re.sub(r'(?<!\w)\*(?!\w)', '', text)
        
        return text.strip()
    
    def _extract_section(self, text: str, section_name: str) -> str:

        patterns = [
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\n[\w\s]+:|===|$)",  # Main pattern
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n[A-Z][A-Z\s]+:|$)",
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\*\*|$)" 
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                result = match.group(1).strip()
                if result:
                    return self._clean_markdown_formatting(result)
        
        return f"{section_name} not found"
    
    def _extract_process_info(self, text: str, prefix: str) -> Tuple[str, float]:

        patterns = [
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]+?)\s*\(([0-9.]+)\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]*?)\s*\(\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*Confidence Score:\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\*\*Confidence Score:\s*([0-9.]+)\*\*",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*\*\*Confidence Score:\s*([0-9.]+)\*\*\s*\)"
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                process_name = match.group(1).strip()
                try:
                    confidence_score = float(match.group(2))
                    process_name = self._clean_markdown_formatting(process_name)
                    process_name = process_name.strip()
                    if process_name and confidence_score >= 0:
                        return process_name, confidence_score
                except ValueError:
                    continue
        
        simple_pattern = rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)(?=\n|$)"
        match = re.search(simple_pattern, text, re.IGNORECASE)
        if match:
            process_name = match.group(1).strip()
            process_name = self._clean_markdown_formatting(process_name)
            process_name = re.sub(r'\s*\([^)]*$', '', process_name)
            process_name = process_name.strip()
            return process_name, 0.0
        
        return f"Unknown Process {prefix} Enrichment", 0.0
    
    def _extract_final_process(self, text: str) -> str:

        match = re.search(r"FINAL PROCESS REASONING:(.*?)(?=\n\n|$)", text, re.DOTALL | re.IGNORECASE)
        if match:
            result = match.group(1).strip()
            return self._clean_markdown_formatting(result)
        return "Final process reasoning not found"
    
    def analyze_community_optimized(self, comm_id: str, genes: List[str]) -> Dict:

        results = {
            "Community": comm_id,
            "Genes": genes,
            "Genes_String": ", ".join(genes)
        }
        
        try:
            enrichment_pathways = self.perform_enrichment(genes)
            results["Enrichment_Pathways"] = enrichment_pathways
            
            all_in_one_prompt = self._create_all_in_one_prompt(genes, enrichment_pathways)
            
            full_analysis = self._make_deepseek_request(all_in_one_prompt)
            results["Full_Analysis"] = full_analysis

            process_with_enrichment, confidence_with_enrichment = self._extract_process_info(full_analysis, "WITH")
            results["Process_With_Enrichment"] = process_with_enrichment
            results["Confidence_With_Enrichment"] = confidence_with_enrichment

            process_without_enrichment, confidence_without_enrichment = self._extract_process_info(full_analysis, "WITHOUT")
            results["Process_Without_Enrichment"] = process_without_enrichment
            results["Confidence_Without_Enrichment"] = confidence_without_enrichment
            
            try:
                results["Pathway_Reasoning_With_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITH ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Pathway_Reasoning_Without_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITHOUT ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Contributing_Genes_With_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITH ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Contributing_Genes_Without_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITHOUT ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Analysis_Text_With_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITH ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Analysis_Text_Without_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITHOUT ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                final_reasoning = self._extract_section(full_analysis, "FINAL PROCESS REASONING")
                results["Final_Process_Reasoning"] = final_reasoning
            except Exception as e:
                results["Final_Process_Reasoning"] = f"Extraction failed: {str(e)}"
            
            if confidence_with_enrichment >= confidence_without_enrichment:
                results["Final_Process"] = process_with_enrichment
                results["Final_Confidence"] = confidence_with_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_With_Enrichment", "Not available")
            else:
                results["Final_Process"] = process_without_enrichment
                results["Final_Confidence"] = confidence_without_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_Without_Enrichment", "Not available")
            
            return results
            
        except Exception as e:
            error_msg = f"Analysis failed: {str(e)}"
            print(f"Error analyzing community {comm_id}: {e}")
            results["Error"] = error_msg
            results.update({
                "Process_With_Enrichment": f"Error: {error_msg}",
                "Confidence_With_Enrichment": 0.0,
                "Process_Without_Enrichment": f"Error: {error_msg}",
                "Confidence_Without_Enrichment": 0.0,
                "Final_Process": f"Error: {error_msg}",
                "Final_Confidence": 0.0,
                "Full_Analysis": f"Error: {error_msg}",
                "Pathway_Reasoning_With_Enrichment": "Error occurred",
                "Pathway_Reasoning_Without_Enrichment": "Error occurred",
                "Contributing_Genes_With_Enrichment": "Error occurred",
                "Contributing_Genes_Without_Enrichment": "Error occurred",
                "Analysis_Text_With_Enrichment": "Error occurred",
                "Analysis_Text_Without_Enrichment": "Error occurred",
                "Final_Process_Reasoning": "Error occurred",
                "Final_Contributing_Genes": "Error occurred"
            })
            return results
    
    def analyze_communities_batch(self, communities: Dict[str, List[str]], batch_size: int = 1) -> List[Dict]:

        all_results = []
        community_items = list(communities.items())
        
        from tqdm import tqdm
        
        progress_bar = tqdm(total=len(community_items), desc="Analyzing communities", unit="community")
        
        for i in range(0, len(community_items), batch_size):
            batch = community_items[i:i+batch_size]
            
            for comm_id, genes in batch:
                result = self.analyze_community_optimized(comm_id, genes)
                all_results.append(result)

                progress_bar.update(1)

                processed = len(all_results)
                total = len(community_items)
                progress_bar.set_description(f"Processed {processed}/{total} communities")
                
                progress_bar.set_postfix({"Current": f"Community {comm_id}", "Genes": len(genes)})
        
        progress_bar.close()
        
        return all_results
    
    def create_detailed_dataframe(self, communities_text: str, batch_size: int = 1) -> pd.DataFrame:

        communities = self.parse_communities(communities_text)
        all_results = self.analyze_communities_batch(communities, batch_size)
        
        full_df = pd.DataFrame(all_results)
        
        requested_columns = [
            "Community",
            "Genes_String",
            "Enrichment_Pathways",
            "Process_With_Enrichment",
            "Confidence_With_Enrichment", 
            "Pathway_Reasoning_With_Enrichment",
            "Contributing_Genes_With_Enrichment",
            "Process_Without_Enrichment",
            "Confidence_Without_Enrichment",
            "Pathway_Reasoning_Without_Enrichment",
            "Contributing_Genes_Without_Enrichment", 
            "Final_Process",
            "Final_Confidence",
            "Final_Contributing_Genes",
            "Final_Process_Reasoning",
            "Full_Analysis"
        ]
        
        available_columns = [col for col in requested_columns if col in full_df.columns]
        return full_df[available_columns]

def run_optimized_analysis(input_text: str, api_key: str = None, detailed_csv: str = None, batch_size: int = 1) -> pd.DataFrame:

    analyzer = OptimizedPathwayAnalyzer()
    analyzer.set_api_key(api_key)
    
    detailed_df = analyzer.create_detailed_dataframe(input_text, batch_size)
    
    if detailed_csv:
        detailed_df.to_csv(detailed_csv, index=False)
    
    return detailed_df

if __name__ == "__main__":
    input_text = """
   Refined Community 1: ['KCNN4', 'C1orf106', 'PKP1', 'RASAL1', 'COCH', 'KRT23', 'TMEM164', 'CDC20', 'TPD52L1', 'SAYSD1', 'BIRC5', 'RAD54B', 'LAD1', 'DLGAP5', 'CDH3', 'LRP12', 'NUF2', 'KIAA1161', 'C9orf40', 'FERMT1', 'KIF20A', 'HENMT1', 'TCF7L1', 'WNT7B', 'ASS1', 'KIF14', 'REXO4', 'DTD1', 'MOSPD1', 'UBE2C', 'PTBP2', 'RDH10', 'PLK1', 'ATF3', 'PNMAL1', 'FAM83D', 'CENPF', 'PLA2G4A', 'PDE8B', 'SGMS2', 'DSC2', 'DIAPH2', 'KRT80', 'SRPK1', 'TTK', 'NCAPG', 'UBE2E3', 'MEST', 'GRB10', 'STMN1']
   """
    
    detailed_df = run_optimized_analysis(
        input_text=input_text,
        api_key=os.environ.get("DEEPSEEK_API_KEY"), 
        detailed_csv="TP53BCAnnotation.csv"
    )
    
    pd.set_option('display.max_colwidth', None)
    print("\nPathway Analysis Results (Clean Text):")

    main_columns = ['Community', 'Process_With_Enrichment', 'Confidence_With_Enrichment', 
                   'Process_Without_Enrichment', 'Confidence_Without_Enrichment', 
                   'Final_Process', 'Final_Confidence']
    available_main_columns = [col for col in main_columns if col in detailed_df.columns]
    print(detailed_df[available_main_columns])
    
    print(f"\nTotal communities analyzed: {len(detailed_df)}")
    print(f"Available columns: {list(detailed_df.columns)}")
    
    if 'Contributing_Genes_With_Enrichment' in detailed_df.columns:
        print(f"\nSample contributing genes (cleaned):")
        for idx, row in detailed_df.head(3).iterrows():
            print(f"Community {row['Community']}: {row.get('Contributing_Genes_With_Enrichment', 'N/A')}")

In [ ]:
import os
import re
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from typing import Dict, List, Any, Optional
from tqdm import tqdm
import os
#os.environ["NCBI_EMAIL"] = ""
#os.environ["NCBI_API_KEY"] = ""

breast_terms = (
    '"Breast Neoplasms"[MeSH Terms] OR '
    '"breast cancer"[All Fields] OR '
    '"breast carcinoma"[All Fields] OR '
    '"breast tumor"[All Fields] OR '
    '"breast tumour"[All Fields]'
)

tp53_terms = (
    '"TP53"[All Fields] OR '
    '"p53"[All Fields] OR '
    '"Tumor Suppressor Protein p53"[MeSH Terms]'
)




def extract_unique_genes_with_communities(
    gene_sets: Dict[str, List[str]]
) -> tuple:

    gene_to_communities: Dict[str, List[str]] = {}

    for community, genes in gene_sets.items():
        for gene in genes:
            gene_to_communities.setdefault(gene, []).append(community)

    unique_genes = list(gene_to_communities.keys())

    print(
        f"Extracted {len(unique_genes)} unique genes "
        f"from {len(gene_sets)} communities"
    )

    return unique_genes, gene_to_communities


def fetch_pmc_full_text(
    pmcid: str,
    email: str,
    api_key: Optional[str] = None,
) -> Optional[str]:

    try:

        params = {
            "db": "pmc",
            "id": pmcid,
            "rettype": "full",
            "retmode": "xml",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=params,
            headers={"User-Agent": f"AML_GeneResearch/1.0 (mailto:{email})"},
            timeout=30,
        )

        if not response.ok:
            return None

        root = ET.fromstring(response.content)

        body = root.find(".//body")

        if body is None:
            return None

        paragraphs = []

        for elem in body.iter():
            if elem.tag in {"p", "title", "sec"}:
                text = "".join(elem.itertext()).strip()

                if text:
                    paragraphs.append(text)

        full_text = "\n\n".join(paragraphs)

        return full_text if full_text else None

    except Exception:
        return None


def parse_pubmed_xml(xml_content: str) -> List[Dict[str, Any]]:

    papers = []

    try:

        root = ET.fromstring(xml_content)

        for article_elem in root.findall(".//PubmedArticle"):

            try:

                paper: Dict[str, Any] = {}

                # PMID
                pmid_elem = article_elem.find(".//PMID")
                paper["pmid"] = (
                    pmid_elem.text if pmid_elem is not None else "Unknown"
                )

                # Title
                title_elem = article_elem.find(".//ArticleTitle")

                if title_elem is not None:
                    title_text = "".join(title_elem.itertext()).strip()
                    paper["title"] = title_text if title_text else "Unknown Title"
                else:
                    paper["title"] = "Unknown Title"

                # Abstract
                abstract_parts = article_elem.findall(".//AbstractText")

                abstract_text = " ".join(
                    [p.text for p in abstract_parts if p.text]
                )

                paper["abstract"] = (
                    abstract_text
                    if abstract_text
                    else "Abstract not available"
                )

                # Journal
                journal_elem = article_elem.find(".//Journal/Title")

                paper["journal"] = (
                    journal_elem.text
                    if journal_elem is not None
                    else "Unknown Journal"
                )

                # ISSN
                issn_elem = article_elem.find(
                    ".//Journal/ISSN[@IssnType='Print']"
                )

                eissn_elem = article_elem.find(
                    ".//Journal/ISSN[@IssnType='Electronic']"
                )

                paper["issn"] = (
                    issn_elem.text if issn_elem is not None else ""
                )

                paper["eissn"] = (
                    eissn_elem.text if eissn_elem is not None else ""
                )

                # Year
                pub_date = article_elem.find(".//PubDate/Year")

                if pub_date is not None:

                    paper["year"] = pub_date.text

                else:

                    medline_date = article_elem.find(".//PubDate/MedlineDate")

                    if medline_date is not None and medline_date.text:

                        year_match = re.search(
                            r"\d{4}",
                            medline_date.text
                        )

                        paper["year"] = (
                            year_match.group(0)
                            if year_match
                            else "Unknown"
                        )

                    else:

                        paper["year"] = "Unknown"

                # Volume
                volume_elem = article_elem.find(".//Volume")

                paper["volume"] = (
                    volume_elem.text
                    if volume_elem is not None
                    else "Unknown"
                )

                # Pages
                pages_elem = article_elem.find(".//MedlinePgn")

                paper["pages"] = (
                    pages_elem.text
                    if pages_elem is not None
                    else "Unknown"
                )

                # Authors
                authors = []

                author_list = article_elem.find(".//AuthorList")

                if author_list is not None:

                    for author in author_list.findall(".//Author"):

                        last = author.find("LastName")
                        inits = author.find("Initials")

                        if last is not None and inits is not None:
                            authors.append(f"{last.text} {inits.text}")

                        elif last is not None:
                            authors.append(last.text)

                paper["authors"] = (
                    ", ".join(authors)
                    if authors
                    else "Unknown Authors"
                )

                # DOI + PMCID
                doi = None
                pmc_id = None

                id_list = article_elem.find(".//ArticleIdList")

                if id_list is not None:

                    for id_elem in id_list.findall(".//ArticleId"):

                        if id_elem.get("IdType") == "doi":
                            doi = id_elem.text

                        elif id_elem.get("IdType") == "pmc":
                            pmc_id = id_elem.text

                paper["doi"] = doi or "DOI not available"
                paper["pmcid"] = pmc_id

                papers.append(paper)

            except Exception:
                continue

    except Exception:
        pass

    return papers

def fetch_pubmed_for_gene(
    gene: str,
    start_year: int = 2010,
    end_year: int = 2026,
    max_results: int = 10,
) -> List[Dict[str, Any]]:

    email = os.environ.get("NCBI_EMAIL")
    api_key = os.environ.get("NCBI_API_KEY")

    if not email:
        raise ValueError("NCBI_EMAIL environment variable not set")

    try:

        gene_terms = (
            f'("{gene}"[Title/Abstract] OR "{gene}"[All Fields])'
        )

        query = (
    f'{gene}[All Fields] AND '
    f'(({breast_terms}) AND ({tp53_terms})) AND '
    f'({start_year}:{end_year}[PDAT])'
)

        headers = {
            "User-Agent": f"AML_GeneResearch/1.0 (mailto:{email})"
        }

        search_params = {
            "db": "pubmed",
            "term": query,
            "retmax": max_results,
            "sort": "relevance",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            search_params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=search_params,
            headers=headers,
            timeout=30,
        )

        if not response.ok:
            return []

        root = ET.fromstring(response.text)

        pmids = [elem.text for elem in root.findall(".//Id")]

        if not pmids:
            return []

        fetch_params = {
            "db": "pubmed",
            "id": ",".join(pmids),
            "retmode": "xml",
            "email": email,
            "tool": "AML_GeneResearch",
        }

        if api_key:
            fetch_params["api_key"] = api_key

        fetch_response = requests.post(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            data=fetch_params,
            headers=headers,
            timeout=60,
        )

        if not fetch_response.ok:
            return []

        papers = parse_pubmed_xml(fetch_response.text)

        for paper in papers:

            pmcid = paper.get("pmcid")

            if pmcid:

                time.sleep(0.35)

                full_text = fetch_pmc_full_text(
                    pmcid,
                    email,
                    api_key,
                )

                paper["full_text"] = (
                    full_text
                    if full_text
                    else "Full text not available via PMC"
                )

            else:

                paper["full_text"] = (
                    "Full text not available via PMC"
                )

        return papers

    except Exception as e:

        print(f"fetch_pubmed_for_gene({gene}) failed: {e}")

        return []


def fetch_pubmed_for_all_unique_genes(
    gene_sets: Dict[str, List[str]],
    start_year: int = 2010,
    end_year: int = 2026,
    max_results_per_gene: int = 10,
) -> pd.DataFrame:

    unique_genes, gene_to_communities = (
        extract_unique_genes_with_communities(gene_sets)
    )

    all_results = []
    genes_with_papers = 0

    for gene in tqdm(
        unique_genes,
        desc="Genes",
        unit="gene",
        ncols=100,
    ):

        try:

            papers = fetch_pubmed_for_gene(
                gene=gene,
                start_year=start_year,
                end_year=end_year,
                max_results=max_results_per_gene,
            )

            if papers:

                genes_with_papers += 1

                for paper in papers:

                    paper["gene"] = gene

                    paper["communities"] = ", ".join(
                        gene_to_communities[gene]
                    )

                    paper["community_count"] = len(
                        gene_to_communities[gene]
                    )

                    all_results.append(paper)

            time.sleep(0.35)

        except Exception as e:

            print(f"Error processing gene {gene}: {e}")

            continue

    total = len(unique_genes)

    print(
        f"\nGenes with papers: "
        f"{genes_with_papers}/{total} "
        f"({genes_with_papers/total*100:.1f}%)"
    )

    print(f"Total papers: {len(all_results)}")

    if not all_results:
        return pd.DataFrame()

    df = pd.DataFrame(all_results)

    column_order = [
        "gene",
        "communities",
        "community_count",
        "title",
        "authors",
        "journal",
        "year",
        "volume",
        "pages",
        "doi",
        "pmid",
        "pmcid",
        "abstract",
        "full_text",
    ]

    df = df[[col for col in column_order if col in df.columns]]

    if "pmid" in df.columns and "gene" in df.columns:

        before = len(df)

        df = df.drop_duplicates(subset=["pmid", "gene"])

        dropped = before - len(df)

        if dropped:
            print(f"Dropped {dropped} duplicate rows")

    return df


ANNOTATION_CSV = "TP53BCAnnotation.csv"
OUTPUT_CSV = "BCTP53_Paper_DB.csv"

START_YEAR = 2010
END_YEAR = 2026
MAX_RESULTS_PER_GENE = 20


annotation_df = pd.read_csv(ANNOTATION_CSV)

annotation_df.columns = [c.strip() for c in annotation_df.columns]

print(f"Loaded {len(annotation_df)} rows")


gene_col = next(
    (
        c for c in annotation_df.columns
        if c in (
            "Genes_String",
            "Contributing_Genes",
            "Genes",
            "Final_Contributing_Genes",
        )
    ),
    None,
)

community_col = next(
    (
        c for c in annotation_df.columns
        if c in (
            "Community",
            "Set_ID",
            "community",
            "index",
        )
    ),
    None,
)

if gene_col is None or community_col is None:
    raise ValueError(
        f"Could not find required columns.\n"
        f"Columns: {list(annotation_df.columns)}"
    )

print(f"Using gene column: {gene_col}")
print(f"Using community column: {community_col}")


gene_sets: Dict[str, List[str]] = {}

for _, row in annotation_df.iterrows():

    community = str(row[community_col])

    raw_genes = str(row[gene_col])

    if raw_genes and raw_genes.lower() != "nan":

        sep = "," if "," in raw_genes else ";"

        genes = [
            g.strip()
            for g in raw_genes.split(sep)
            if g.strip()
        ]

        gene_sets[community] = genes

print(f"Built {len(gene_sets)} gene-set communities")

already_fetched = set()

if os.path.exists(OUTPUT_CSV):

    existing_df = pd.read_csv(OUTPUT_CSV)

    already_fetched = set(
        existing_df["gene"].dropna().unique()
    )

    print(
        f"Resuming from existing database "
        f"({len(already_fetched)} genes already fetched)"
    )

    gene_sets_filtered = {}

    for community, genes in gene_sets.items():

        remaining = [
            g for g in genes
            if g not in already_fetched
        ]

        if remaining:
            gene_sets_filtered[community] = remaining

    gene_sets = gene_sets_filtered


papers_df = fetch_pubmed_for_all_unique_genes(
    gene_sets=gene_sets,
    start_year=START_YEAR,
    end_year=END_YEAR,
    max_results_per_gene=MAX_RESULTS_PER_GENE,
)


if not papers_df.empty:

    if os.path.exists(OUTPUT_CSV) and already_fetched:

        papers_df.to_csv(
            OUTPUT_CSV,
            mode="a",
            header=False,
            index=False,
        )

        print(
            f"\nAppended {len(papers_df)} rows to {OUTPUT_CSV}"
        )

    else:

        papers_df.to_csv(
            OUTPUT_CSV,
            index=False,
        )

        print(
            f"\nSaved {len(papers_df)} rows to {OUTPUT_CSV}"
        )

else:

    print("\nNo papers retrieved")


print(f"\nDone! AML database saved to: {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import json
import os
import re
import time
import requests
from typing import List, Dict, Any, Optional, Tuple
from tqdm import tqdm

UNKNOWN_PROCESS_LABEL = "unknown process"


def _is_unknown(process_name: str) -> bool:
    return process_name.strip().lower() == UNKNOWN_PROCESS_LABEL


class GeneSetValidator:
    def __init__(
        self,
        gene_sets_path: str,
        papers_csv: str = "AML_Paper_DB.csv",
        api_key: Optional[str] = None,
        output_path: str = "validated_gene_sets.csv",
    ):
        self.gene_sets_path = gene_sets_path
        self.papers_csv     = papers_csv
        self.output_path    = output_path
        self.api_key        = api_key or os.environ.get("DEEPSEEK_API_KEY")

        if not self.api_key:
            print("Warning: No DeepSeek API key provided.")

        self.gene_sets_df      = None
        self.papers_df         = None
        self.validated_results = []

        # ── Load Clarivate JIF journal list ───────────────────────────────────
        try:
            jif_df = pd.read_csv("journals_filtered_JIF_ge_4.csv", sep=",")
            jif_df.columns = [c.strip() for c in jif_df.columns]

            self._hq_issn_set  = set()
            self._hq_eissn_set = set()
            self.TOP_JOURNALS  = []

            for _, jr in jif_df.iterrows():
                issn  = str(jr.get("ISSN",  "")).strip().replace("-", "").upper()
                eissn = str(jr.get("eISSN", "")).strip().replace("-", "").upper()
                name  = str(jr.get("Journal name", "")).strip()

                if issn  and issn  != "NAN": self._hq_issn_set.add(issn)
                if eissn and eissn != "NAN": self._hq_eissn_set.add(eissn)
                if name:                     self.TOP_JOURNALS.append(name)

            self._top_journal_set_normalized = {
                self.normalize_journal(j) for j in self.TOP_JOURNALS
            }

            print(f"Loaded {len(jif_df)} high-quality journals from Clarivate JIF CSV "
                  f"({len(self._hq_issn_set)} ISSNs, {len(self._hq_eissn_set)} eISSNs)")

        except FileNotFoundError:
            print("Warning: journals_filtered_JIF_ge_4.csv not found. No quality filter applied.")
            self._hq_issn_set                = set()
            self._hq_eissn_set               = set()
            self.TOP_JOURNALS                = []
            self._top_journal_set_normalized = set()

    # ── Helpers ───────────────────────────────────────────────────────────────

    def normalize_journal(self, name) -> str:
        if pd.isna(name):
            return ""
        name = str(name).lower()
        name = name.split(" : ")[0].split(" - ")[0]
        name = re.sub(r"[^\w\s]", "", name)
        name = re.sub(r"\s+", " ", name)
        return name.strip()

    def _normalize_id(self, val) -> str:
        return str(val).strip().replace("-", "").upper() if pd.notna(val) else ""

    def _first_author_surname(self, authors) -> str:
        if pd.isna(authors):
            return "Unknown"
        s = str(authors).strip()
        if not s or s.lower() in ("nan", "no authors"):
            return "Unknown"
        first = re.split(r"[;,]|\band\b", s)[0].strip()
        parts = [p for p in first.split() if not re.fullmatch(r"[A-Z]\.?[A-Z]?\.?", p)]
        return " ".join(parts) if parts else first

    def _is_high_quality(self, row) -> bool:
        issn = self._normalize_id(row.get("issn"))
        if issn and issn in self._hq_issn_set:
            return True
        eissn = self._normalize_id(row.get("eissn"))
        if eissn and eissn in self._hq_eissn_set:
            return True
        norm = self.normalize_journal(row.get("journal", ""))
        if norm and norm in self._top_journal_set_normalized:
            return True
        return False

    def _load_papers_for_genes(self, genes: List[str]) -> pd.DataFrame:
        if self.papers_df is None or self.papers_df.empty:
            return pd.DataFrame()

        genes_upper = {g.upper() for g in genes if g}
        mask   = self.papers_df["gene"].str.upper().isin(genes_upper)
        subset = self.papers_df[mask].copy()

        if subset.empty:
            return subset

        if "full_text" not in subset.columns:
            subset["full_text"] = "Full text not available via PMC"

        subset["matched_journal"] = subset.apply(
            lambda row: "HQ" if self._is_high_quality(row) else None, axis=1
        )

        hq  = subset["matched_journal"].notna().sum()
        tot = len(subset)
        print(f"  Paper filter: {hq}/{tot} high-quality papers for gene set")
        return subset


    def load_data(self):
        print("Loading gene sets data...")
        self.gene_sets_df = pd.read_csv(self.gene_sets_path)
        self.gene_sets_df.columns = [col.strip() for col in self.gene_sets_df.columns]
        print(f"Loaded {len(self.gene_sets_df)} gene sets from {self.gene_sets_path}")

        print(f"Loading AML paper database from {self.papers_csv}...")
        if not os.path.exists(self.papers_csv):
            print(f"Paper CSV '{self.papers_csv}' not found.")
            self.papers_df = pd.DataFrame()
        else:
            self.papers_df = pd.read_csv(self.papers_csv, low_memory=False)
            self.papers_df.columns = [col.strip() for col in self.papers_df.columns]
            if "gene" not in self.papers_df.columns:
                print("'gene' column not found in paper CSV - gene filtering disabled.")
                self.papers_df = pd.DataFrame()
            else:
                self.papers_df["gene"] = self.papers_df["gene"].astype(str).str.strip()
                print(f"Loaded {len(self.papers_df)} papers ({self.papers_df['gene'].nunique()} unique genes)")


    def extract_genes_from_set(self, gene_set_row: pd.Series) -> List[str]:
        genes = []
        possible_gene_columns = [
            "Genes_String", "Contributing_Genes", "Genes", "Final_Contributing_Genes",
        ]
        for col in possible_gene_columns:
            if col in gene_set_row and pd.notna(gene_set_row[col]):
                gene_data = gene_set_row[col]
                if isinstance(gene_data, list):
                    genes = gene_data
                elif isinstance(gene_data, str):
                    if "," in gene_data:
                        genes = [g.strip() for g in gene_data.split(",")]
                    elif ";" in gene_data:
                        genes = [g.strip() for g in gene_data.split(";")]
                    else:
                        genes = [gene_data.strip()]
                break
        return [g for g in genes if g and g.strip()]

    def extract_gene_related_abstracts(
        self,
        genes: List[str],
        papers_df: pd.DataFrame,
        max_papers: int = 50,
    ) -> Tuple[List[Dict], List[str]]:

        if isinstance(genes, str):
            genes = [g.strip() for g in genes.split(",")]

        genes_set = {g.upper() for g in genes if g}

        hq_papers = []

        for _, paper in papers_df.iterrows():

            if pd.isna(paper.get("matched_journal")):
                continue

            abstract  = str(paper.get("abstract",  ""))
            full_text = str(paper.get("full_text", ""))

            if abstract  in ("nan", "Abstract not available", ""):
                abstract = ""
            if full_text in ("nan", "Full text not available via PMC", ""):
                full_text = ""

            if not abstract and not full_text:
                continue

            search_corpus = (abstract + " " + full_text).strip()
            mentioned_genes = [
                g for g in genes
                if g and re.search(rf"\b{re.escape(g)}\b", search_corpus, re.IGNORECASE)
            ]
            gene_from_col = str(paper.get("gene", "")).strip()
            if gene_from_col and gene_from_col.upper() in genes_set:
                if gene_from_col not in mentioned_genes:
                    mentioned_genes.append(gene_from_col)

            if not mentioned_genes:
                continue

            truncated_abstract  = (abstract[:500]  + "...") if len(abstract)  > 500  else abstract
            truncated_full_text = (full_text[:3000] + "...") if len(full_text) > 3000 else full_text

            hq_papers.append({
                "title":           paper.get("title",   "No title"),
                "authors":         paper.get("authors", "No authors"),
                "first_author":    self._first_author_surname(paper.get("authors")),
                "journal":         paper.get("journal", "No journal"),
                "year":            paper.get("year",    "Unknown"),
                "abstract":        truncated_abstract,
                "full_text":       truncated_full_text,
                "doi":             paper.get("doi",     "No DOI"),
                "pmid":            paper.get("pmid",    "No PMID"),
                "genes_mentioned": mentioned_genes,
                "gene_count":      len(mentioned_genes),
            })

        hq_papers.sort(key=lambda x: x["gene_count"], reverse=True)
        relevant_papers = hq_papers[:max_papers]

        top_journal_papers_used = [
            "Paper {n} ({fa} et al.) | {title} | {journal} ({year}) | PMID: {pmid} | DOI: {doi}".format(
                n=idx,
                fa=p["first_author"],
                title=str(p["title"]).replace("|", "/"),
                journal=p["journal"],
                year=p["year"],
                pmid=p["pmid"],
                doi=p["doi"],
            )
            for idx, p in enumerate(relevant_papers, 1)
        ]

        return relevant_papers, top_journal_papers_used


    def create_validation_prompt(
        self,
        gene_set_id: str,
        genes: List[str],
        process_with_enrichment: str,
        confidence_with_enrichment: float,
        process_without_enrichment: str,
        confidence_without_enrichment: float,
        analysis_with_enrichment: str,
        analysis_without_enrichment: str,
        relevant_papers: List[Dict],
    ) -> str:

        genes_str      = ", ".join(genes) if isinstance(genes, list) else str(genes)
        papers_section = ""
        for i, paper in enumerate(relevant_papers, 1):
            papers_section += (
                f"PAPER {i} ({paper['first_author']} et al.):\n"
                f"Title: {paper['title']}\n"
                f"Authors: {paper['authors']}\n"
                f"Journal: {paper['journal']} ({paper['year']})\n"
                f"Genes Mentioned: {', '.join(paper['genes_mentioned'])}\n"
                f"Abstract: {paper['abstract']}\n"
                f"\n"
            )

        return f"""You are a scientific expert in genomics and bioinformatics tasked with validating gene set analysis results using ONLY the provided literature.

GENE SET ID: {gene_set_id}
GENES: {genes_str}

ORIGINAL ANALYSIS WITH ENRICHMENT:
Process Name: {process_with_enrichment}
Original Confidence Score: {confidence_with_enrichment}
Analysis: {analysis_with_enrichment[:1000]}...

ORIGINAL ANALYSIS WITHOUT ENRICHMENT:
Process Name: {process_without_enrichment}
Original Confidence Score: {confidence_without_enrichment}
Analysis: {analysis_without_enrichment[:1000]}...

PROVIDED LITERATURE (USE ONLY THESE STUDIES):
{papers_section}

VALIDATION TASK:
Based STRICTLY on the provided literature above, evaluate both analyses and provide updated confidence scores.

**ABSOLUTE REQUIREMENT FOR PAPER CITATIONS:**
Whenever you reference a paper anywhere in your response, always use the exact label given in that paper's PAPER header:
"Paper X (FirstAuthorLastName et al.)"
For example: "Paper 3 (Nakamura et al.) shows..." or "supported by Paper 1 (Chen et al.) and Paper 4 (Okafor et al.)"
Copy the surname EXACTLY as it appears in the PAPER header - do NOT re-derive it from the Authors line.
Do NOT use bare numbers like "Paper 3" alone, and do NOT spell out full titles or complete author lists in-line - the first-author-et-al form is sufficient everywhere in the response.

CRITICAL REQUIREMENTS:
1. Use ONLY the provided papers - do not add external knowledge
2. For each process, check if the genes are supported by the literature
3. Provide updated confidence scores based on evidence strength
4. Select the better-supported process as the final choice
5. **MANDATORY: When referencing papers by number anywhere in your analysis text, always use the "Paper X (FirstAuthor et al.)" label from the PAPER header - never bare numbers, never full citations.**
6. The final selected process MUST be either:
   (a) exactly one of the two original process names, OR
   (b) "Neither process" ONLY if both updated confidence scores (Updated Confidence With Enrichment, Updated Confidence Without Enrichment) are <= 0.05.

7. The final confidence MUST follow:
   - If a process is selected: Final Confidence = the updated confidence of that selected process.
   - If "Neither process" is selected: Final Confidence = max(Updated Confidence With Enrichment, Updated Confidence Without Enrichment).

FORMAT YOUR RESPONSE EXACTLY AS FOLLOWS (plain text, no markdown bold):

VALIDATION OF ENRICHMENT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_with_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

VALIDATION OF DIRECT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_without_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

FINAL PROCESS SELECTION:
Selected Process: [Choose the better-supported process name]
Final Confidence: [The updated confidence score for your selected process]
Selection Reasoning: [Explain why this process has stronger literature support]

CONFLICT ANALYSIS:
Review all papers for contradictory evidence about gene functions, pathway assignments, or experimental results.
CONFLICTING_EVIDENCE_FOUND: [TRUE/FALSE]
CONFLICT_DESCRIPTION: [Brief description of any conflicts found, or "No conflicts detected"]

VALIDATION ANALYSIS TEXT:
[Comprehensive summary of all changes made, reasoning for confidence adjustments, and evidence from the provided studies that led to the final process selection.
Use "Paper X (FirstAuthor et al.)" format for any paper references.]"""

    # ── DeepSeek API ──────────────────────────────────────────────────────────

    def call_deepseek_api(self, prompt: str) -> str:
        if not self.api_key:
            raise ValueError("DeepSeek API key is required.")

        api_url = "https://api.deepseek.com/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
        }
        data = {
            "model": "deepseek-chat",
            "messages": [
                {
                    "role":    "system",
                    "content": (
                        "You are a scientific expert in genomics and bioinformatics "
                        "specializing in gene set analysis validation. "
                        "Base your analysis strictly on the provided literature."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            "temperature": 0,
            "max_tokens":  8000,
        }

        max_retries = 3
        retry_delay = 5

        for attempt in range(max_retries):
            try:
                response = requests.post(api_url, headers=headers, json=data, timeout=180)

                if response.status_code == 200:
                    return response.json()["choices"][0]["message"]["content"]

                elif response.status_code == 429:
                    wait_time = int(response.headers.get("Retry-After", retry_delay * 2))
                    print(f"Rate limited. Waiting {wait_time}s...")
                    time.sleep(wait_time)

                elif 500 <= response.status_code < 600:
                    print(f"Server error {response.status_code}. Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                    retry_delay *= 2

                else:
                    raise Exception(f"API Error {response.status_code}: {response.text}")

            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{max_retries}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"API timeout after {max_retries} attempts")

            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"Request failed after {max_retries} attempts: {e}")

        raise Exception(f"Failed to get a valid response after {max_retries} attempts")

    # ── Response parser ───────────────────────────────────────────────────────

    def parse_llm_response(self, response: str) -> Dict[str, Any]:
        results = {}

        m = re.search(
            r"VALIDATION OF ENRICHMENT ANALYSIS:.*?Updated Confidence:\**\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_with_enrichment_after"] = float(m.group(1))

        m = re.search(
            r"VALIDATION OF DIRECT ANALYSIS:.*?Updated Confidence:\**\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_without_enrichment_after"] = float(m.group(1))

        m = re.search(r"Selected Process:\**\s*(.+?)(?=\n|$)", response)
        if m:
            results["final_process"] = m.group(1).strip().strip("*").strip()

        m = re.search(r"Final Confidence:\**\s*([\d.]+)", response)
        if m:
            results["final_confidence"] = float(m.group(1))

        m = re.search(
            r"CONFLICTING_EVIDENCE_FOUND:\**\s*(TRUE|FALSE)", response, re.IGNORECASE
        )
        results["conflicting_evidence_found"] = (
            m.group(1).upper() == "TRUE" if m else False
        )

        m = re.search(
            r"CONFLICT_DESCRIPTION:\**\s*(.+?)(?=\n\s*\n|VALIDATION ANALYSIS TEXT:|\Z)",
            response, re.DOTALL,
        )
        results["conflict_description"] = m.group(1).strip() if m else "No conflicts detected"

        m = re.search(r"VALIDATION ANALYSIS TEXT:\**\s*(.+?)\Z", response, re.DOTALL)
        if m:
            results["validation_analysis_text"] = m.group(1).strip()

        return results

    # ── Single gene-set validation ────────────────────────────────────────────

    def validate_gene_set(self, gene_set_row: pd.Series) -> Dict[str, Any]:
        set_id = str(
            gene_set_row.get("Community", "")
            or gene_set_row.get("Set_ID", "")
            or gene_set_row.get("community", "")
            or gene_set_row.get("index", "")
        )

        genes = self.extract_genes_from_set(gene_set_row)
        if not genes:
            print(f"  No genes found for set {set_id}")
            return self._error_result(set_id, "No genes found for this set")

        process_with    = str(gene_set_row.get("Process_With_Enrichment", "")
                              or gene_set_row.get("Final_Process", ""))
        conf_with       = float(gene_set_row.get("Confidence_With_Enrichment", 0)
                                or gene_set_row.get("Final_Confidence", 0))
        process_without = str(gene_set_row.get("Process_Without_Enrichment", ""))
        conf_without    = float(gene_set_row.get("Confidence_Without_Enrichment", 0))

        analysis_with    = str(gene_set_row.get("Analysis_Text_With_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_With_Enrichment", "")
                               or gene_set_row.get("Full_Analysis", ""))
        analysis_without = str(gene_set_row.get("Analysis_Text_Without_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_Without_Enrichment", ""))

        # ── Unknown process guard ─────────────────────────────────────────────
        with_is_unknown    = _is_unknown(process_with)
        without_is_unknown = _is_unknown(process_without)

        if with_is_unknown and without_is_unknown:
            print(f"  Both processes are 'unknown process' for set {set_id} - skipping.")
            return self._both_unknown_result(set_id, genes, gene_set_row)

        if with_is_unknown:
            print(f"  'With enrichment' is unknown process for set {set_id} - zeroing out, validating direct only.")
            process_with  = UNKNOWN_PROCESS_LABEL
            conf_with     = 0.0
            analysis_with = "Unknown process - not validated."

        if without_is_unknown:
            print(f"  'Without enrichment' is unknown process for set {set_id} - zeroing out, validating enrichment only.")
            process_without  = UNKNOWN_PROCESS_LABEL
            conf_without     = 0.0
            analysis_without = "Unknown process - not validated."

        papers_df = self._load_papers_for_genes(genes)

        relevant_papers, top_journal_info = self.extract_gene_related_abstracts(
            genes, papers_df, max_papers=50
        )

        if not relevant_papers:
            print(f"  No matching high-quality papers for set {set_id}")
            return self._no_papers_result(set_id, genes, gene_set_row)

        prompt = self.create_validation_prompt(
            gene_set_id=set_id,
            genes=genes,
            process_with_enrichment=process_with,
            confidence_with_enrichment=conf_with,
            process_without_enrichment=process_without,
            confidence_without_enrichment=conf_without,
            analysis_with_enrichment=analysis_with,
            analysis_without_enrichment=analysis_without,
            relevant_papers=relevant_papers,
        )

        os.makedirs("prompts", exist_ok=True)
        with open(f"prompts/prompt_{set_id}.txt", "w", encoding="utf-8") as f:
            f.write(prompt)

        try:
            response = self.call_deepseek_api(prompt)

            os.makedirs("responses", exist_ok=True)
            with open(f"responses/response_{set_id}.txt", "w", encoding="utf-8") as f:
                f.write(response)

            parsed = self.parse_llm_response(response)

            conf_with_after    = parsed.get("confidence_with_enrichment_after",   conf_with)
            conf_without_after = parsed.get("confidence_without_enrichment_after", conf_without)
            if with_is_unknown:
                conf_with_after = 0.0
            if without_is_unknown:
                conf_without_after = 0.0

            return {
                "Set_ID":                                set_id,
                "Genes":                                 genes,
                "Process_With_Enrichment_Original":      process_with,
                "Process_Without_Enrichment_Original":   process_without,
                "Confidence_With_Enrichment_Before":     conf_with,
                "Confidence_Without_Enrichment_Before":  conf_without,
                "Confidence_With_Enrichment_After":      conf_with_after,
                "Confidence_Without_Enrichment_After":   conf_without_after,
                "Final_Process":                         parsed.get("final_process",   process_with),
                "Final_Confidence":                      parsed.get("final_confidence", conf_with),
                "Validation_Analysis_Text":              parsed.get("validation_analysis_text", "No analysis provided"),
                "Conflicting_Evidence_Found":            parsed.get("conflicting_evidence_found", False),
                "Conflict_Description":                  parsed.get("conflict_description", "No conflicts detected"),
                "Top_Journal_Papers_Used":               " || ".join(top_journal_info) if top_journal_info else "No top journal papers found",
                "Total_Papers_Found":                    len(relevant_papers),
            }

        except Exception as e:
            print(f"  API error for set {set_id}: {e}")
            return self._error_result(
                set_id, f"API error: {e}", genes, gene_set_row, top_journal_info
            )


    def validate_all_gene_sets(self):
        if self.gene_sets_df is None:
            self.load_data()

        results    = []
        total_sets = len(self.gene_sets_df)
        print(f"\nStarting validation of {total_sets} gene sets...")

        pbar = tqdm(
            total=total_sets,
            desc="Validating gene sets",
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
            ncols=100,
        )

        for i, (_, row) in enumerate(self.gene_sets_df.iterrows()):
            set_id = str(
                row.get("Community", "")
                or row.get("Set_ID", "")
                or row.get("community", "")
                or i
            )
            try:
                result = self.validate_gene_set(row)
                results.append(result)
            except Exception as e:
                print(f"\nUnexpected error on set {set_id}: {e}")

            pbar.update(1)
            pbar.set_postfix_str(f"Set {set_id}")

            if (i + 1) % 5 == 0:
                self.validated_results = results
                self.save_results(f"{self.output_path}.partial")

        pbar.close()
        self.validated_results = results
        self.save_results(self.output_path)
        print(f"\nValidation complete! Processed {len(results)} gene sets")
        return results

    # ── Save ──────────────────────────────────────────────────────────────────

    def save_results(self, output_path: str = None):
        if not self.validated_results:
            print("No results to save")
            return

        path = output_path or self.output_path
        df   = pd.DataFrame(self.validated_results)

        required_columns = [
            "Set_ID", "Genes",
            "Process_With_Enrichment_Original", "Process_Without_Enrichment_Original",
            "Confidence_With_Enrichment_Before", "Confidence_Without_Enrichment_Before",
            "Confidence_With_Enrichment_After",  "Confidence_Without_Enrichment_After",
            "Final_Process", "Final_Confidence",
            "Validation_Analysis_Text",
            "Conflicting_Evidence_Found", "Conflict_Description",
            "Top_Journal_Papers_Used", "Total_Papers_Found",
        ]
        for col in required_columns:
            if col not in df.columns:
                df[col] = None
        df = df[required_columns]
        df.to_csv(path, index=False)
        print(f"Saved {len(df)} validated results to {path}")


    def generate_summary_report(self) -> str:
        if not self.validated_results:
            return "No validation results available"

        total = len(self.validated_results)

        conflict_count     = sum(1 for r in self.validated_results if r.get("Conflicting_Evidence_Found"))
        total_paper_counts = [r.get("Total_Papers_Found", 0) for r in self.validated_results]
        final_confidences  = [r.get("Final_Confidence", 0)   for r in self.validated_results]

        conf_delta_enrich = []
        conf_delta_direct = []
        for r in self.validated_results:
            be, ae = r.get("Confidence_With_Enrichment_Before", 0),   r.get("Confidence_With_Enrichment_After", 0)
            bd, ad = r.get("Confidence_Without_Enrichment_Before", 0), r.get("Confidence_Without_Enrichment_After", 0)
            if be > 0 and ae > 0: conf_delta_enrich.append(ae - be)
            if bd > 0 and ad > 0: conf_delta_direct.append(ad - bd)

        lines = [
            "Gene Set Validation Summary Report",
            "=" * 50,
            f"Total gene sets processed: {total}",
            "",
            "CONFLICT ANALYSIS:",
            f"  With conflicts:    {conflict_count} ({conflict_count/total*100:.1f}%)",
            f"  Without conflicts: {total - conflict_count} ({(total-conflict_count)/total*100:.1f}%)",
            "",
            "PAPER DISCOVERY (all papers are high-quality by construction):",
            f"  Avg papers/set:   {sum(total_paper_counts)/len(total_paper_counts):.2f}",
            f"  Sets with papers: {sum(1 for c in total_paper_counts if c > 0)} ({sum(1 for c in total_paper_counts if c > 0)/total*100:.1f}%)",
            "",
            "CONFIDENCE:",
            f"  Avg final confidence: {sum(final_confidences)/len(final_confidences):.2f}",
            f"  Range: {min(final_confidences):.2f} - {max(final_confidences):.2f}",
        ]
        if conf_delta_enrich:
            lines.append(f"  Avg enrichment confidence delta: {sum(conf_delta_enrich)/len(conf_delta_enrich):+.3f}")
        if conf_delta_direct:
            lines.append(f"  Avg direct confidence delta:     {sum(conf_delta_direct)/len(conf_delta_direct):+.3f}")

        return "\n".join(lines)


    def _error_result(
        self,
        set_id: str,
        error_msg: str,
        genes: List[str] = None,
        row: pd.Series = None,
        top_journal_info: List[str] = None,
    ) -> Dict[str, Any]:
        genes = genes or []
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      row.get("Process_With_Enrichment", "") if row is not None else "",
            "Process_Without_Enrichment_Original":   row.get("Process_Without_Enrichment", "") if row is not None else "",
            "Confidence_With_Enrichment_Before":     row.get("Confidence_With_Enrichment", 0) if row is not None else 0,
            "Confidence_Without_Enrichment_Before":  row.get("Confidence_Without_Enrichment", 0) if row is not None else 0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         "ERROR",
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              error_msg,
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Error occurred during validation",
            "Top_Journal_Papers_Used":               " || ".join(top_journal_info) if top_journal_info else "Error - no papers processed",
            "Total_Papers_Found":                    len(top_journal_info) if top_journal_info else 0,
        }

    def _no_papers_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        conf_with = float(row.get("Confidence_With_Enrichment", 0) or row.get("Final_Confidence", 0))
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Process_Without_Enrichment_Original":   str(row.get("Process_Without_Enrichment", "")),
            "Confidence_With_Enrichment_Before":     conf_with,
            "Confidence_Without_Enrichment_Before":  float(row.get("Confidence_Without_Enrichment", 0)),
            "Confidence_With_Enrichment_After":      conf_with,
            "Confidence_Without_Enrichment_After":   float(row.get("Confidence_Without_Enrichment", 0)),
            "Final_Process":                         str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Final_Confidence":                      conf_with,
            "Validation_Analysis_Text":              "No high-quality papers found for this gene set",
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "No high-quality papers available",
            "Top_Journal_Papers_Used":               "No high-quality papers found",
            "Total_Papers_Found":                    0,
        }

    def _both_unknown_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      UNKNOWN_PROCESS_LABEL,
            "Process_Without_Enrichment_Original":   UNKNOWN_PROCESS_LABEL,
            "Confidence_With_Enrichment_Before":     0,
            "Confidence_Without_Enrichment_Before":  0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         UNKNOWN_PROCESS_LABEL,
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              "Both processes are unknown - validation skipped.",
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Both processes unknown - no validation performed",
            "Top_Journal_Papers_Used":               "N/A",
            "Total_Papers_Found":                    0,
        }



def main():
    gene_sets_path = "TP53BCAnnotation.csv"
    papers_csv     = "BCTP53_Paper_DB.csv"
    output_path    = "TP53BCValidation.csv"
    
    
    print("AML Gene Set Validation Pipeline")
    print("=" * 50)

    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        print("No DEEPSEEK_API_KEY env var found.")
        print("Set it with:  export DEEPSEEK_API_KEY=your_key")
        return

    print("API key found")

    validator = GeneSetValidator(
        gene_sets_path=gene_sets_path,
        papers_csv=papers_csv,
        api_key=api_key,
        output_path=output_path,
    )

    validator.load_data()

    test_mode = False

    if test_mode:
        print("\nTest mode - validating first gene set only")
        result = validator.validate_gene_set(validator.gene_sets_df.iloc[0])
        validator.validated_results = [result]
        validator.save_results("test_validation.csv")
        for k, v in result.items():
            print(f"  {k}: {len(v) if isinstance(v, list) else v}")
    else:
        validator.validate_all_gene_sets()

    report = validator.generate_summary_report()
    print("\n" + report)
    with open("validation_summary.txt", "w") as f:
        f.write(report)

    print("\nDone!")


if __name__ == "__main__":
    main()